# Text-to-Outfit — Notebook 2 (v3): query encoder + fine-tuning de la última capa del Transformer

Requiere `t2o_samples_with_embeddings_v3.pt` del Notebook 1 (v3), que ya trae `train`/`valid`/
`test` separados y, por cada outfit, `outfit_emb` (congelado), `prelast_seq` y
`key_padding_mask` (para el fine-tuning).

**Qué se entrena en esta versión:**
- El `QueryEncoder` (GloVe -> MLP -> 128-d), igual que antes.
- La **última capa del Transformer** del Recommendation Engine (`last_layer`), inicializada
  desde el checkpoint y descongelada -- con un learning rate mucho más bajo que el del
  `QueryEncoder`, para moverla con cuidado y no destruir lo ya aprendido.

**Qué sigue congelado:** todo lo anterior a la última capa (CNN de imagen, `TextEncoder` de
ítem, `CategoricalEncoder`, `TypeAwareGate`, primera capa del Transformer) -- por eso el
Notebook 1 pudo cachear `prelast_seq` una sola vez, y aquí no hace falta tocar imágenes en
ningún momento del entrenamiento.


In [ ]:
import json
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Preprocesado de texto (idéntico a versiones anteriores)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize(text):
    if not isinstance(text, str) or text == '':
        return []
    return word_tokenize(text)


lemmatizer = WordNetLemmatizer()


def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)


def lemmatize(tokens):
    return [lemmatizer.lemmatize(t, get_wordnet_pos(t)) for t in tokens]


def remove_duplicates(tokens):
    return list(dict.fromkeys(tokens))


def preprocess_for_glove(text):
    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    tokens = lemmatize(tokens)
    tokens = remove_duplicates(tokens)
    return tokens

## 2. Cargar GloVe

In [ ]:
GLOVE_PATH = "./polyvore_kaggle/polyvore_outfits/glove/wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt"  # AJUSTA
EMBED_DIM = 100


def load_glove(glove_path):
    embeddings = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split()
            word = " ".join(parts[:-EMBED_DIM])
            vector = np.asarray(parts[-EMBED_DIM:], dtype=np.float32)
            embeddings[word] = vector
    return embeddings


glove = load_glove(GLOVE_PATH)
print(f"Vectores GloVe cargados: {len(glove)}")

## 3. Texto libre -> vector GloVe

In [ ]:
def text_to_glove_vector(text, glove=glove):
    tokens = preprocess_for_glove(text)
    vecs = [glove[t] for t in tokens if t in glove]
    if not vecs:
        return np.zeros(EMBED_DIM, dtype=np.float32), True
    vec = np.mean(vecs, axis=0)
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    return vec.astype(np.float32), False

## 4. Cargar `t2o_samples_with_embeddings_v3.pt` y preparar `group_id` + vector de query

El fichero del Notebook 1 v3 ya trae los splits separados (con Capas 1+4 ya aplicadas) y,
por outfit, `outfit_emb` / `prelast_seq` / `key_padding_mask`. Aquí solo añadimos el vector
GloVe de cada texto y el `group_id` para el multi-positivo (igual que en la v2).

In [ ]:
T2O_CACHE_PATH = "./polyvore_kaggle/polyvore_outfits/checkpoints_RE/t2o_samples_with_embeddings_filtered_solocapauno.pt"  # ajusta

cache = torch.load(T2O_CACHE_PATH)
t2o_train, t2o_valid, t2o_test = cache["train"], cache["valid"], cache["test"]
print(f"Cargados -> train={len(t2o_train)}  valid={len(t2o_valid)}  test={len(t2o_test)}")


def prepare_split(samples):
    kept = []
    for s in samples:
        vec, is_oov = text_to_glove_vector(s["text"])
        if is_oov:
            continue
        s["query_glove_vec"] = torch.from_numpy(vec)
        kept.append(s)
    return kept


t2o_train = prepare_split(t2o_train)
t2o_valid = prepare_split(t2o_valid)
t2o_test  = prepare_split(t2o_test)

# t2o_samples = torch.load(T2O_CACHE_PATH)
# print(f"Muestras cargadas: {len(t2o_samples)}")

# n_full_oov = 0
# for s in tqdm(t2o_samples, desc="Vectorizando texto con GloVe"):
#     vec, is_oov = text_to_glove_vector(s["text"])
#     s["query_glove_vec"] = torch.from_numpy(vec)
#     if is_oov:
#         n_full_oov += 1

# print(f"Muestras con texto totalmente OOV (vector cero): {n_full_oov}")

# t2o_samples = [s for s in t2o_samples if not torch.all(s["query_glove_vec"] == 0)]
# print(f"Muestras utilizables tras descartar OOV totales: {len(t2o_samples)}")

# t2o_train = [s for s in t2o_samples if s["split"] == "train"]
# t2o_valid = [s for s in t2o_samples if s["split"] == "valid"]
# t2o_test  = [s for s in t2o_samples if s["split"] == "test"]


def assign_group_ids(samples):
    text_to_gid = {}
    for s in samples:
        if s["text"] not in text_to_gid:
            text_to_gid[s["text"]] = len(text_to_gid)
        s["group_id"] = text_to_gid[s["text"]]
    return samples, len(text_to_gid)


t2o_train, n_groups_train = assign_group_ids(t2o_train)
t2o_valid, n_groups_valid = assign_group_ids(t2o_valid)
t2o_test,  n_groups_test  = assign_group_ids(t2o_test)

print(f"Tras filtrar OOV totales -> train={len(t2o_train)} ({n_groups_train} textos únicos)  "
      f"valid={len(t2o_valid)} ({n_groups_valid} textos únicos)  "
      f"test={len(t2o_test)} ({n_groups_test} textos únicos)")

## 5. Dataset + DataLoader

Cada muestra devuelve ahora también `prelast_seq` y `key_padding_mask` (necesarios para poder
recalcular el forward de la última capa del Transformer durante el entrenamiento), además de
`outfit_emb` congelado (útil para comparar contra la versión no-fine-tuned), `query_glove_vec`
y `group_id`.

In [ ]:
class T2OQueryDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return (s["outfit_emb"], s["prelast_seq"], s["key_padding_mask"],
                s["query_glove_vec"], s["group_id"])


def make_t2o_loader(samples, batch_size=128, shuffle=True):
    ds = T2OQueryDataset(samples)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)


BATCH_SIZE = 128

t2o_train_loader = make_t2o_loader(t2o_train, batch_size=BATCH_SIZE, shuffle=True)
t2o_valid_loader = make_t2o_loader(t2o_valid, batch_size=BATCH_SIZE, shuffle=False)
t2o_test_loader  = make_t2o_loader(t2o_test,  batch_size=BATCH_SIZE, shuffle=False)

## 6. Última capa del Transformer (descongelada) + `QueryEncoder`

Reconstruimos la clase `TransformerEncoderBlock` (idéntica a la del Recommendation Engine, para
poder cargar los pesos del checkpoint) e inicializamos una instancia **solo con la última capa**,
cargando sus pesos exactos desde `best_ce.pt`. Esta es la única parte del Recommendation Engine
que se entrena en este notebook.

In [ ]:
D_MODEL, N_HEADS, FFN_DIM, DROPOUT = 128, 4, 256, 0.1


class TransformerEncoderBlock(nn.Module):
    """Misma definición que en el Recommendation Engine -- necesaria para poder cargar
    los pesos de la última capa desde el checkpoint original."""
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
                                                dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        ffn_out = self.ffn(x)
        return self.norm2(x + self.drop2(ffn_out))


CKPT_PATH_CE = "./polyvore_kaggle/polyvore_outfits/checkpoints_RE/best_ce.pt"  # ajusta si hace falta
ckpt = torch.load(CKPT_PATH_CE, map_location=device)
full_state_dict = ckpt["model_state_dict"]

# Extraemos solo los pesos de la ÚLTIMA capa (encoder_layers.1.* en un modelo de 2 capas)
last_layer_prefix = "encoder_layers.1."
last_layer_state = {
    k[len(last_layer_prefix):]: v
    for k, v in full_state_dict.items()
    if k.startswith(last_layer_prefix)
}
assert len(last_layer_state) > 0, "No se encontraron pesos de encoder_layers.1 -- revisa el prefijo"

last_layer = TransformerEncoderBlock(D_MODEL, N_HEADS, FFN_DIM, DROPOUT).to(device)
last_layer.load_state_dict(last_layer_state)
for p in last_layer.parameters():
    p.requires_grad = True  # <-- descongelada, esto es el fine-tuning

print(f"Última capa cargada e inicializada desde el checkpoint. "
      f"Parámetros entrenables: {sum(p.numel() for p in last_layer.parameters()):,}")


class QueryEncoder(nn.Module):
    def __init__(self, input_dim=EMBED_DIM, hidden_dim=256, output_dim=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, glove_vec):
        x = self.net(glove_vec)
        return F.normalize(x, p=2, dim=-1)


query_encoder = QueryEncoder().to(device)
print(f"Parámetros del query encoder: {sum(p.numel() for p in query_encoder.parameters()):,}")

## 7. Pérdida contrastiva multi-positivo (igual que en la v2)

In [ ]:
class ContrastiveHead(nn.Module):
    def __init__(self, init_temperature=0.07):
        super().__init__()
        self.log_temp = nn.Parameter(torch.tensor(np.log(1.0 / init_temperature)))

    def forward(self):
        return self.log_temp.exp().clamp(max=100.0)


contrastive_head = ContrastiveHead().to(device)


def masked_nce(logits, positive_mask):
    log_prob = F.log_softmax(logits, dim=1)
    pos_count = positive_mask.sum(dim=1).clamp(min=1.0)
    loss_per_row = -(log_prob * positive_mask).sum(dim=1) / pos_count
    return loss_per_row.mean()


def contrastive_loss(outfit_emb, query_emb, group_ids, temperature):
    logits = outfit_emb @ query_emb.T * temperature
    positive_mask = (group_ids.unsqueeze(1) == group_ids.unsqueeze(0)).float()
    loss_o2q = masked_nce(logits, positive_mask)
    loss_q2o = masked_nce(logits.T, positive_mask.T)
    return (loss_o2q + loss_q2o) / 2

## 8. Forward completo del outfit (recalculando la última capa, ahora entrenable)

En vez de usar directamente `outfit_emb` (congelado, precalculado), esta función recalcula el
paso por `last_layer` a partir de `prelast_seq` -- eso es lo que permite que el gradiente fluya
hacia sus pesos durante el entrenamiento. `prelast_seq` en sí sigue siendo un tensor fijo
(nunca requiere gradiente), así que el coste extra es solo el de una capa de Transformer sobre
secuencias muy cortas -- nada de CNN ni de imágenes de por medio.

In [ ]:
def compute_outfit_emb_finetuned(prelast_seq, key_padding_mask):
    """prelast_seq: (B, MAX_LEN+1, 128) -- ya en el dispositivo.
    key_padding_mask: (B, MAX_LEN+1) bool."""
    x = last_layer(prelast_seq, key_padding_mask=key_padding_mask)
    return F.normalize(x[:, 0, :], p=2, dim=-1)

## 9. Métrica de evaluación: Recall@k multi-positivo (usa la última capa fine-tuned)

In [ ]:
@torch.no_grad()
def evaluate_recall_at_k(samples, ks=(1, 5, 10, 30, 50), use_finetuned=True, batch_size=256):
    query_encoder.eval()
    last_layer.eval()

    all_outfit_embs, all_glove_vecs, all_group_ids = [], [], []
    for i in range(0, len(samples), batch_size):
        batch = samples[i:i + batch_size]
        glove_vecs = torch.stack([s["query_glove_vec"] for s in batch]).to(device)
        all_glove_vecs.append(glove_vecs)
        all_group_ids.extend([s["group_id"] for s in batch])

        if use_finetuned:
            prelast = torch.stack([s["prelast_seq"] for s in batch]).to(device)
            mask    = torch.stack([s["key_padding_mask"] for s in batch]).to(device)
            outfit_embs = compute_outfit_emb_finetuned(prelast, mask)
        else:
            outfit_embs = torch.stack([s["outfit_emb"] for s in batch]).to(device)
        all_outfit_embs.append(outfit_embs)

    outfit_embs = torch.cat(all_outfit_embs, dim=0)          # (N, 128)
    glove_vecs  = torch.cat(all_glove_vecs, dim=0)            # (N, 100)
    group_ids   = torch.tensor(all_group_ids, device=device)  # (N,)

    query_embs = query_encoder(glove_vecs)
    sims = query_embs @ outfit_embs.T

    n = sims.shape[0]
    relevant_mask = (group_ids.unsqueeze(1) == group_ids.unsqueeze(0))

    sorted_idx = sims.argsort(dim=1, descending=True)
    ranks_matrix = torch.empty_like(sorted_idx)
    rank_values = torch.arange(1, n + 1, device=device).unsqueeze(0).expand(n, -1)
    ranks_matrix.scatter_(1, sorted_idx, rank_values)

    ranks_masked = torch.where(relevant_mask, ranks_matrix, torch.full_like(ranks_matrix, n + 1))
    best_rank = ranks_masked.min(dim=1).values.float()

    results = {f"Recall@{k}": (best_rank <= k).float().mean().item() for k in ks}
    results["MRR"] = (1.0 / best_rank).mean().item()
    results["MedianRank"] = best_rank.median().item()

    query_encoder.train()
    last_layer.train()
    return results

## 10. Bucle de entrenamiento -- learning rates diferenciados

`QueryEncoder` (nuevo, necesita moverse mucho) usa `LR_QUERY=1e-3`. `last_layer` (ya
preentrenada, solo queremos ajustarla con cuidado) usa `LR_FINETUNE=1e-5` -- dos órdenes de
magnitud más bajo, para minimizar el riesgo de olvido catastrófico.

In [ ]:
LR_QUERY     = 1e-3
LR_FINETUNE  = 1e-5
WEIGHT_DECAY = 1e-4
EPOCHS       = 100
PATIENCE     = 10

optimizer = torch.optim.AdamW([
    {"params": query_encoder.parameters(),     "lr": LR_QUERY},
    {"params": contrastive_head.parameters(),  "lr": LR_QUERY},
    {"params": last_layer.parameters(),        "lr": LR_FINETUNE},
], weight_decay=WEIGHT_DECAY)

CKPT_PATH_OUT = "./polyvore_kaggle/polyvore_outfits/checkpoints_RE/best_query_encoder_finetuned_v3.pt"

best_recall5 = -1.0
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    query_encoder.train()
    last_layer.train()
    train_losses = []

    for outfit_emb, prelast_seq, key_padding_mask, glove_vec, group_id in t2o_train_loader:
        prelast_seq      = prelast_seq.to(device)
        key_padding_mask = key_padding_mask.to(device)
        glove_vec        = glove_vec.to(device)
        group_id         = group_id.to(device)

        outfit_emb_ft = compute_outfit_emb_finetuned(prelast_seq, key_padding_mask)
        query_emb = query_encoder(glove_vec)
        temperature = contrastive_head()
        loss = contrastive_loss(outfit_emb_ft, query_emb, group_id, temperature)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    val_metrics = evaluate_recall_at_k(t2o_valid, use_finetuned=True)
    train_loss_mean = np.mean(train_losses)

    print(f"Época {epoch:3d} | loss={train_loss_mean:.4f} | "
          f"val R@1={val_metrics['Recall@1']:.3f} R@5={val_metrics['Recall@5']:.3f} "
          f"R@10={val_metrics['Recall@10']:.3f} medRank={val_metrics['MedianRank']:.1f}")

    if val_metrics["Recall@5"] > best_recall5:
        best_recall5 = val_metrics["Recall@5"]
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "query_encoder_state_dict": query_encoder.state_dict(),
            "last_layer_state_dict": last_layer.state_dict(),
            "contrastive_head_state_dict": contrastive_head.state_dict(),
            "val_recall5": best_recall5,
        }, CKPT_PATH_OUT)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping en época {epoch} (sin mejora en {PATIENCE} épocas)")
            break

print(f"\nMejor Recall@5 en validación: {best_recall5:.3f}")

## 11. Evaluación final en test -- comparando fine-tuned vs. congelado

Cargamos el mejor checkpoint y evaluamos **dos veces** sobre el mismo test: una con la última
capa fine-tuned (`use_finetuned=True`) y otra usando directamente el `outfit_emb` original
100% congelado (`use_finetuned=False`, usando el `QueryEncoder` ya entrenado igualmente). Así
se ve de forma directa cuánto ha aportado el fine-tuning, no solo el número final.

In [ ]:
CKPT_PATH_OUT = "./polyvore_kaggle/polyvore_outfits/checkpoints_RE/best_query_encoder_finetuned_v3.pt"

best_ckpt = torch.load(CKPT_PATH_OUT, map_location=device)
query_encoder.load_state_dict(best_ckpt["query_encoder_state_dict"])
last_layer.load_state_dict(best_ckpt["last_layer_state_dict"])
contrastive_head.load_state_dict(best_ckpt["contrastive_head_state_dict"])

test_metrics_ft = evaluate_recall_at_k(t2o_test, use_finetuned=True)
print("Resultados en test -- ÚLTIMA CAPA FINE-TUNED:")
for k, v in test_metrics_ft.items():
    print(f"  {k}: {v:.3f}")

print()
test_metrics_frozen = evaluate_recall_at_k(t2o_test, use_finetuned=False)
print("Resultados en test -- outfit_emb 100% CONGELADO (mismo query_encoder entrenado):")
for k, v in test_metrics_frozen.items():
    print(f"  {k}: {v:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- Datos extraídos de los experimentos (Zero-shot vs Fine-tuned) ---
epochs_zs = list(range(1, 28))
loss_zs = [
    4.6842, 4.5178, 4.4362, 4.3712, 4.3195, 4.2705, 4.2357, 4.1988, 4.1734, 4.1453,
    4.1161, 4.0977, 4.0771, 4.0626, 4.0403, 4.0283, 4.0139, 3.9980, 3.9878, 3.9728,
    3.9615, 3.9542, 3.9435, 3.9331, 3.9209, 3.9092, 3.9037,
]
r5_zs = [
    4.7, 6.9, 6.4, 8.0, 4.4, 5.4, 7.9, 8.9, 7.9, 8.9,
    9.5, 8.6, 8.2, 8.5, 11.3, 8.9, 9.3, 10.4, 8.4, 9.3,
    9.6, 10.6, 10.5, 8.7, 11.0, 10.4, 8.7,
]  # Recall@5 en % (x100 respecto a las proporciones originales)

epochs_ft = list(range(1, 30))
loss_ft = [
    4.6873, 4.5166, 4.4229, 4.3558, 4.3006, 4.2522, 4.2099, 4.1730, 4.1379, 4.1030,
    4.0783, 4.0505, 4.0314, 4.0094, 3.9907, 3.9714, 3.9510, 3.9365, 3.9200, 3.9074,
    3.8928, 3.8774, 3.8690, 3.8569, 3.8463, 3.8354, 3.8264, 3.8146, 3.8010,
]
r5_ft = [
    2.5, 4.3, 6.4, 5.6, 8.8, 8.8, 10.6, 9.8, 7.6, 9.8,
    9.4, 10.4, 10.1, 10.6, 9.6, 11.4, 10.3, 11.2, 12.2, 11.0,
    10.7, 11.3, 11.0, 10.1, 11.3, 10.5, 11.2, 10.9, 10.1,
]  # Recall@5 en %

stop_zs = epochs_zs[-1]  # 27
stop_ft = epochs_ft[-1]  # 29
best_ep_zs, best_r5_zs = epochs_zs[r5_zs.index(max(r5_zs))], max(r5_zs)
best_ep_ft, best_r5_ft = epochs_ft[r5_ft.index(max(r5_ft))], max(r5_ft)

# --- Configuración global tipográfica y de estilo para la memoria ---
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 12,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10.5,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
        "axes.edgecolor": "0.25",
        "figure.dpi": 300,
    }
)

COLOR_ZS, COLOR_FT = "#1f77b4", "#ff7f0e"  # Azul y Naranja institucionales


def mark_stop(ax, epoch, color, y_frac, label, va="top"):
    ax.axvline(epoch, color=color, lw=1.2, ls=":", alpha=0.85)
    ylo, yhi = ax.get_ylim()
    ax.text(
        epoch - 0.25,
        ylo + (yhi - ylo) * y_frac,
        label,
        fontsize=8.5,
        color=color,
        rotation=90,
        ha="right",
        va=va,
        clip_on=True,
    )


# --- Creación de la figura 1x2 unificada ---
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(12.0, 4.2))

# =========================================================
# Panel (a): Pérdida de entrenamiento -- ZS vs FT
# =========================================================
ax_a.plot(
    epochs_zs, loss_zs, color=COLOR_ZS, ls="-", marker="o", ms=4.0, lw=1.8, label="Zero-shot",
)
ax_a.plot(
    epochs_ft, loss_ft, color=COLOR_FT, ls="-", marker="^", ms=4.0, lw=1.8, label="Fine-tuned",
)
ax_a.set_xlabel("Época")
ax_a.set_ylabel(r"Pérdida contrastiva ($\mathcal{L}$)")
max_ep = max(stop_zs, stop_ft)
ax_a.set_xticks(range(1, max_ep + 1, 3))
ax_a.set_xlim(0.5, max_ep + 1.0)

mark_stop(ax_a, stop_zs, COLOR_ZS, 0.92, "Early stopping (ZS)")
mark_stop(ax_a, stop_ft, COLOR_FT, 0.92, "Early stopping (FT)")

ax_a.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9)
ax_a.set_title("(a) Evolución de la pérdida de entrenamiento")

# =========================================================
# Panel (b): Recall@5 de validación -- ZS vs FT (en %)
# =========================================================
ax_b.plot(
    epochs_zs, r5_zs, color=COLOR_ZS, ls="--", marker="o", ms=4.0, lw=1.8, label="Zero-shot (val)",
)
ax_b.plot(
    epochs_ft, r5_ft, color=COLOR_FT, ls="--", marker="^", ms=4.0, lw=1.8, label="Fine-tuned (val)",
)

# Destacar óptimos con estrella
ax_b.scatter(
    [best_ep_zs], [best_r5_zs], color=COLOR_ZS, s=110, marker="*",
    edgecolor="black", linewidth=0.8, zorder=5,
)
# ax_b.annotate(
#     f"Óptimo ZS (Ép. {best_ep_zs})\nR@5 = {best_r5_zs:.1f}%",
#     xy=(best_ep_zs, best_r5_zs),
#     xytext=(best_ep_zs - 10.5, best_r5_zs + 1.5),
#     fontsize=9.0, color=COLOR_ZS,
#     arrowprops=dict(arrowstyle="->", color=COLOR_ZS, lw=1.0),
# )

ax_b.scatter(
    [best_ep_ft], [best_r5_ft], color=COLOR_FT, s=110, marker="*",
    edgecolor="black", linewidth=0.8, zorder=5,
)
# ax_b.annotate(
#     f"Óptimo FT (Ép. {best_ep_ft})\nR@5 = {best_r5_ft:.1f}%",
#     xy=(best_ep_ft, best_r5_ft),
#     xytext=(best_ep_ft - 8.0, best_r5_ft - 3.5),
#     fontsize=9.0, color=COLOR_FT,
#     arrowprops=dict(arrowstyle="->", color=COLOR_FT, lw=1.0),
# )

ax_b.set_xlabel("Época")
ax_b.set_ylabel("Recall@5 (%)")
ax_b.set_xticks(range(1, max_ep + 1, 3))
ax_b.set_xlim(0.5, max_ep + 1.0)
ax_b.set_ylim(0.0, 15.5)
ax_b.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))

mark_stop(ax_b, stop_zs, COLOR_ZS, 0.05, "Early stopping (ZS)", va="bottom")
mark_stop(ax_b, stop_ft, COLOR_FT, 0.05, "Early stopping (FT)", va="bottom")

ax_b.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.9)
ax_b.set_title("(b) Rendimiento en recuperación (Recall@5)")

plt.tight_layout()

# Guardar figura unificada
import os

os.makedirs("figuras", exist_ok=True)
fig.savefig("figuras/CLIP4Outfit_ZS_vs_FT.png", bbox_inches="tight", dpi=300)
fig.savefig("figuras/CLIP4Outfit_ZS_vs_FT.pdf", bbox_inches="tight")
plt.show()

## 12. Inferencia: recuperar outfits a partir de una consulta libre

In [ ]:
@torch.no_grad()
def retrieve_outfits(query_text, candidate_samples=t2o_test, top_k=5, use_finetuned=True):
    query_encoder.eval()
    last_layer.eval()

    vec, is_oov = text_to_glove_vector(query_text)
    if is_oov:
        print("Aviso: la consulta no tiene ninguna palabra reconocida por GloVe.")
    glove_vec = torch.from_numpy(vec).unsqueeze(0).to(device)
    query_emb = query_encoder(glove_vec)

    if use_finetuned:
        prelast = torch.stack([s["prelast_seq"] for s in candidate_samples]).to(device)
        mask    = torch.stack([s["key_padding_mask"] for s in candidate_samples]).to(device)
        outfit_embs = compute_outfit_emb_finetuned(prelast, mask)
    else:
        outfit_embs = torch.stack([s["outfit_emb"] for s in candidate_samples]).to(device)

    sims = (query_emb @ outfit_embs.T).squeeze(0)
    top_scores, top_idx = sims.topk(top_k)

    results = []
    for score, idx in zip(top_scores.tolist(), top_idx.tolist()):
        s = candidate_samples[idx]
        results.append({"set_id": s["set_id"], "text": s["text"], "score": score})
    return results


for r in retrieve_outfits("formal office outfit neutral tones"):
    score = r["score"]
    set_id = r["set_id"]
    text = r["text"]
    print(f"{score:.3f}  {set_id:>10}  {text}")
